# **I. Tiền xử lý chung**

**1. Kiểm tra dữ liệu gốc**

In [ ]:
# ---Nạp thư viện---
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr

# ---Xác định thư mục dự án và nạp mô đun tiền xử lý---
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

project_path = str(PROJECT_ROOT.resolve())
if project_path not in sys.path:
    sys.path.insert(0, project_path)

import src.preprocessing as preprocessing

# ---Đọc bảng ứng dụng và tạo bản sao để xử lý---
apps_path = PROJECT_ROOT / "data/raw/googleplaystore.csv"
apps_raw = pd.read_csv(apps_path)
apps = apps_raw.copy()

print(f"Initial rows: {len(apps):,}")

**2. Kiểm tra và loại bản ghi lệch cột**  

Nhóm loại bản ghi có dấu hiệu lệch cột vì nhiều giá trị không phù hợp với ý nghĩa của thuộc tính và chưa có căn cứ khôi phục chính xác. Sau đó, nhóm loại các dòng trùng lặp hoàn toàn, giữ lần xuất hiện đầu tiên. Các bản ghi chỉ trùng tên ứng dụng được giữ lại để xem xét theo từng câu hỏi phân tích.

In [ ]:
# ---Hiển thị bản ghi có dấu hiệu lệch cột---
malformed_mask = preprocessing.malformed_app_mask(apps)
print("Suspected malformed rows:")
display(apps.loc[malformed_mask])

# ---Loại bản ghi lệch cột và báo cáo số dòng---
rows_before = len(apps)
apps = apps.loc[~malformed_mask].copy()

print(f"Rows before: {rows_before:,}")
print(f"Rows removed: {rows_before - len(apps):,}")
print(f"Rows after: {len(apps):,}")

Nhóm xác định và loại một bản ghi của ứng dụng Life Made WI-Fi Touchscreen Photo Frame, có nhiều giá trị không phù hợp với ý nghĩa của cột. Số dòng giảm từ 10.841 xuống 10.840.

**3. Loại bản sao hoàn toàn**  



In [ ]:
# ---Loại bản sao hoàn toàn và giữ lần xuất hiện đầu tiên---
rows_before = len(apps)
duplicate_count = apps.duplicated().sum()
apps = apps.drop_duplicates(keep="first").copy()

print(f"Rows before: {rows_before:,}")
print(f"Exact duplicates removed: {duplicate_count:,}")
print(f"Rows after: {len(apps):,}")
print(f"Remaining exact duplicates: {apps.duplicated().sum():,}")

Nhóm loại 483 bản sao giống nhau hoàn toàn, giữ lại bản ghi xuất hiện đầu tiên. Bảng còn 10.357 dòng và không còn dòng trùng lặp hoàn toàn.

**4. Kiểm tra các bản ghi trùng tên còn lại**

In [ ]:
# ---Thống kê các tên ứng dụng còn xuất hiện nhiều lần---
repeated_names = apps["App"].duplicated(keep=False)
extra_rows = apps["App"].duplicated().sum()
repeated_app_count = apps.loc[repeated_names, "App"].nunique()

name_summary = pd.Series({
    "Remaining rows": len(apps),
    "Distinct app names": apps["App"].nunique(),
    "Extra rows with repeated app names": extra_rows,
    "App names appearing more than once": repeated_app_count
})
display(name_summary)

Bảng còn 9.659 tên ứng dụng phân biệt. Trong đó, 523 tên xuất hiện nhiều lần, tạo ra 698 bản ghi thêm ngoài lần xuất hiện đầu tiên. Nhóm chưa loại các bản ghi này vì cùng tên không đồng nghĩa với thông tin hoàn toàn giống nhau; cách xử lý sẽ được xác định theo đơn vị phân tích của từng câu hỏi.

**5. Chuyển đổi các cột số**  

Nhóm sẽ thực hiện chuyển đổi các cột số như sau:  
- Giữ nguyên cột `Rating` vì đã có kiểu float64
- `Reviews` và `Installs` sang kiểu Int64
- `Price` sang float64  
- Giữ cột `Size` gốc dưới dạng `string`, thêm `Size_MB` kiểu `float64` để tính toán và `Size_varies` kiểu `boolean` để nhận diện placeholder  

Trước khi chuyển đổi, nhóm loại dấu phẩy và dấu + trong `Installs`, cùng ký hiệu $ trong `Price`. Giá trị `Installs` sau chuyển đổi vẫn là cận dưới của lượt cài đặt.

In [ ]:
rows_before = len(apps)

# ---Chuyển Rating và Reviews sang kiểu số---
rating = pd.to_numeric(apps["Rating"], errors="raise")
rating = rating.astype("float64")
reviews = pd.to_numeric(apps["Reviews"], errors="raise")
reviews = reviews.astype("Int64")

# ---Bỏ ký hiệu trong Installs và giữ ý nghĩa cận dưới---
install_text = apps["Installs"].astype("string")
install_text = install_text.str.replace(",", "", regex=False)
install_text = install_text.str.replace("+", "", regex=False)
installs = pd.to_numeric(install_text, errors="raise")
installs = installs.astype("Int64")

# ---Bỏ ký hiệu đô la và chuyển Price sang kiểu số---
price_text = apps["Price"].astype("string")
price_text = price_text.str.replace("$", "", regex=False)
price = pd.to_numeric(price_text, errors="raise")
price = price.astype("float64")

# ---Phân biệt kích thước thiếu và kích thước tùy thiết bị---
size_text = apps["Size"].astype("string").str.strip()
size_varies = (size_text == "Varies with device").fillna(False)
size_missing = size_text.isna() | (size_text == "").fillna(False)
size_parts = size_text.str.extract(r"^(\d+(?:\.\d+)?)([Mk])$")
unexpected_size = (
    ~size_missing & ~size_varies & size_parts[0].isna()
)

if unexpected_size.any():
    display(apps.loc[unexpected_size, ["App", "Size"]])
    raise ValueError("Unrecognized Size values remain.")

# ---Đổi kích thước sang MB theo quy ước k chia 1024---
size_value = pd.to_numeric(size_parts[0], errors="raise")
size_factor = size_parts[1].map({"M": 1.0, "k": 1 / 1024})
size_mb = (size_value * size_factor).astype("float64")

# ---Cập nhật bảng sau khi các phép chuyển đổi thành công---
apps["Rating"] = rating
apps["Reviews"] = reviews
apps["Installs"] = installs
apps["Price"] = price
apps["Size_MB"] = size_mb
apps["Size_varies"] = size_varies

# ---Kiểm tra kiểu dữ liệu và miền giá trị của năm cột số---
numeric_cols = ["Rating", "Reviews", "Size_MB", "Installs", "Price"]
numeric_values = apps[numeric_cols]
conversion_summary = pd.DataFrame({
    "Dtype": numeric_values.dtypes.astype(str),
    "Missing count": numeric_values.isna().sum(),
    "Negative count": (numeric_values < 0).sum(),
    "Min": numeric_values.min(),
    "Max": numeric_values.max()
})

print(f"Rows before: {rows_before:,}")
print(f"Rows after: {len(apps):,}")
display(conversion_summary)

invalid_rating = (
    apps["Rating"].notna() & ~apps["Rating"].between(1, 5)
)
nonpositive_size = (apps["Size_MB"] <= 0).sum()
print(f"Ratings outside 1-5: {invalid_rating.sum():,}")
print(f"Missing original Size: {size_missing.sum():,}")
print(f"Size varies with device: {size_varies.sum():,}")
print(f"Nonpositive Size_MB: {nonpositive_size:,}")

preview_cols = [
    "App", "Rating", "Reviews", "Size", "Size_MB",
    "Size_varies", "Installs", "Price"
]
display(apps[preview_cols].head())

**Nhận xét:**  
- Các cột số đã có kiểu dữ liệu đúng như dự kiến. `Reviews`, `Installs` và `Price` không có giá trị thiếu hoặc âm   
- `Rating` còn 1.465 giá trị thiếu; các điểm quan sát được đều nằm trong thang 1–5  
- `Size_MB` có 1.526 giá trị NaN, tương ứng hoàn toàn với các bản ghi Varies with device. Cột `Size` gốc không thiếu, cờ `Size_varies` giúp nhận diện các trường hợp kích thước chưa xác định  
- Không phát hiện kích thước nhỏ hơn hoặc bằng 0. Bước chuyển đổi không loại thêm bản ghi  

**6. Chuyển đổi cột thời gian `Last Updated`**  

Nhóm chuyển kiểu dữ liệu cột `Last Updated` từ chuỗi sang kiểu `datetime64[ns]` để hỗ trợ so sánh và trích xuất thông tin ngày tháng. Cột này biểu diễn lần cập nhật cuối của ứng dụng, không phải ngày phát hành hoặc ngày thu thập dữ liệu.

In [ ]:
rows_before = len(apps)
missing_before = apps["Last Updated"].isna().sum()

# ---Chuyển ngày theo định dạng gốc, ví dụ January 7, 2018---
last_updated = pd.to_datetime(
    apps["Last Updated"], format="%B %d, %Y", errors="raise"
)
apps["Last Updated"] = last_updated.astype("datetime64[ns]")

# ---Kiểm tra kiểu dữ liệu, giá trị thiếu và khoảng thời gian---
print(f"Rows before: {rows_before:,}")
print(f"Rows after: {len(apps):,}")
print(f"Dtype: {apps['Last Updated'].dtype}")
print(f"Missing values before: {missing_before:,}")
print(f"Missing values after: {apps['Last Updated'].isna().sum():,}")
print(f"Earliest date: {apps['Last Updated'].min():%d/%m/%Y}")
print(f"Latest date: {apps['Last Updated'].max():%d/%m/%Y}")
display(apps[["App", "Last Updated"]].head())

Cột `Last Updated` đã được chuyển sang kiểu `datetime64[ns]`, không phát sinh giá trị thiếu. Ngày cập nhật nhỏ nhất là 21/05/2010, lớn nhất là 08/08/2018. Số dòng giữ nguyên ở 10.357.

**7. Xử lý `Type` thiếu và kiểm tra với `Price`**

In [ ]:
rows_before = len(apps)

# ---Nhận diện và hiển thị các bản ghi thiếu Type---
type_text = apps["Type"].astype("string").str.strip()
missing_type = type_text.isna() | (type_text == "").fillna(False)
print("Rows with missing Type before processing:")
display(apps.loc[missing_type, ["App", "Type", "Price", "Installs"]])

# ---Suy ra Type còn thiếu từ giá hợp lệ---
infer_free = missing_type & (apps["Price"] == 0)
infer_paid = missing_type & (apps["Price"] > 0)
apps.loc[infer_free, "Type"] = "Free"
apps.loc[infer_paid, "Type"] = "Paid"

# ---Kiểm tra nhãn thiếu, nhãn lạ và mâu thuẫn với Price---
type_after = apps["Type"].astype("string").str.strip()
missing_after = (
    type_after.isna() | (type_after == "").fillna(False)
)
invalid_type = ~missing_after & ~type_after.isin(["Free", "Paid"])
conflict = (
    ((type_after == "Free") & (apps["Price"] > 0))
    | ((type_after == "Paid") & (apps["Price"] == 0))
).fillna(False)

type_summary = pd.Series({
    "Missing Type before": missing_type.sum(),
    "Inferred Free": infer_free.sum(),
    "Inferred Paid": infer_paid.sum(),
    "Missing Type after": missing_after.sum(),
    "Invalid Type labels": invalid_type.sum(),
    "Type-Price conflicts": conflict.sum()
})
display(type_summary)

print("Rows with invalid Type or Type-Price conflicts:")
display(apps.loc[invalid_type | conflict, ["App", "Type", "Price"]])
print(f"Rows before: {rows_before:,}")
print(f"Rows after: {len(apps):,}")

Bản ghi "Command & Conquer: Rivals" thiếu `Type` và có `Price` = 0, nên nhóm suy ra nhãn Free. Sau xử lý, cột `Type` không còn giá trị thiếu hoặc nhãn không hợp lệ. Nhóm cũng không phát hiện thêm bản ghi có `Type` mâu thuẫn với `Price`. Do đó số dòng được giữ nguyên.

Các giá trị thiếu của `Rating`, `Current Ver` và `Android Ver` được giữ trong bảng chung. Việc lọc theo điểm đánh giá hoặc tạo nhóm `Unknown` sẽ được thực hiện riêng khi câu hỏi cần đến.

**8. Chuyển kiểu các cột phân loại và văn bản**  

Ở phần data exploration, nhóm có một số đề xuất về việc chuyển đổi các cột biểu diễn nhãn sang kiểu `category` và các cột văn bản sang kiểu `string`. Mỗi tổ hợp trong `Genres` tiếp tục được giữ như một nhãn riêng. Cụ thể:    
- `Category`, `Type`, `Content Rating`, `Genres` → `category`
- `App`, `Size`, `Current Ver`, `Android Ver` → `string`

Giữ nguyên các nhãn tổ hợp, kể cả `Education;Education`; chưa gộp nhãn hiếm hoặc chuẩn hóa các thành phần trong bước tiền xử lý chung.

In [ ]:
rows_before = len(apps)
categorical_cols = ["Category", "Type", "Content Rating", "Genres"]
string_cols = ["App", "Size", "Current Ver", "Android Ver"]
converted_cols = categorical_cols + string_cols

# ---Ghi nhận trạng thái trước khi chuyển kiểu---
dtype_before = apps[converted_cols].dtypes.astype(str)
missing_before = apps[converted_cols].isna().sum()
distinct_before = apps[converted_cols].nunique()

# ---Chuyển các cột phân loại và văn bản---
for col in categorical_cols:
    apps[col] = apps[col].astype("category")

for col in string_cols:
    apps[col] = apps[col].astype("string")

# ---Đối chiếu kiểu dữ liệu, số giá trị thiếu và số nhãn---
conversion_summary = pd.DataFrame({
    "Dtype before": dtype_before,
    "Dtype after": apps[converted_cols].dtypes.astype(str),
    "Missing before": missing_before,
    "Missing after": apps[converted_cols].isna().sum(),
    "Distinct before": distinct_before,
    "Distinct after": apps[converted_cols].nunique()
})
display(conversion_summary)
print(f"Rows before: {rows_before:,}")
print(f"Rows after: {len(apps):,}")

**9. Kiểm tra lại cấu trúc dữ liệu của các cột**

In [ ]:
# ---Đặt lại chỉ số liên tục và xem cấu trúc bảng---
apps = apps.reset_index(drop=True)
apps.info()

Sau tiền xử lý, bảng có 10.357 dòng và 15 cột, gồm 13 cột gốc và hai cột mới `Size_MB`, `Size_varies`. Các cột số, phân loại, văn bản và ngày tháng đã có kiểu dữ liệu phù hợp với kế hoạch

# **II. Đặt câu hỏi và phân tích**

Mỗi thành viên phụ trách hai câu hỏi. Mỗi câu hỏi không dùng ML cần đáp ứng ít nhất hai tiêu chí chiều sâu; các câu hỏi của nhóm cùng bao phủ ít nhất bốn tiêu chí. Các tiêu chí gồm kết hợp dữ liệu, tái cấu trúc dữ liệu hoặc phân tích thời gian, tạo biến dẫn xuất, kiểm soát yếu tố gây nhiễu, lượng hóa độ bất định và kiểm tra độ bền của kết quả.

### **2.1. Chọn mức giá tải cho ứng dụng trả phí**

**Người thực hiện:** Phạm Hoàng Tuấn Anh

**Câu hỏi**

Trong các ứng dụng trả phí, nhóm có giá tải đến 2 USD hay nhóm có giá trên 2 USD có tỷ lệ đạt cận dưới 10.000 lượt cài cao hơn? Thứ tự giữa hai nhóm có giống nhau khi xét chung và khi so sánh trong từng danh mục không?

**Động lực**

Khi đã quyết định thu phí ngay lúc tải, nhà phát triển cần một mốc tham khảo để chọn mức giá và đặt kỳ vọng về quy mô tiếp cận trong danh mục của mình. So sánh hai khoảng giá cho biết ở nhóm giá nào ứng dụng thường đạt mốc lượt cài hơn trong dữ liệu lịch sử.

**Các tiêu chí chiều sâu**

- **D3** — Tạo biến chỉ báo cận dưới lượt cài đạt ít nhất 10.000 từ cột `Installs`. Nhóm ứng dụng theo hai khoảng giá để tính tỷ lệ đạt mốc.
- **D4** — Giữ các danh mục có đủ ứng dụng ở cả hai khoảng giá. Trên cùng tập ứng dụng đó, đối chiếu tỷ lệ chung với tỷ lệ trong từng danh mục và báo số quan sát ở mỗi nhóm.

**Phạm vi diễn giải**

Giá trong dữ liệu là giá niêm yết tại thời điểm thu thập. Chênh lệch quan sát được là thông tin tham khảo về lượt cài. Việc đánh giá tác động của đổi giá hoặc chọn giá tối đa hóa doanh thu cần dữ liệu về lịch sử giá và giao dịch.

#### **1. Chuẩn bị dữ liệu và tạo biến (D3, D4)**

1. Giữ các bản ghi `Type == "Paid"`. Kiểm tra `Price` dương và các cột `Price`, `Installs`, `Category` có giá trị.
2. Giữ một bản ghi cho mỗi cặp `App`–`Category`. Ưu tiên số lượt đánh giá cao hơn, rồi ngày cập nhật mới hơn. Nếu cả hai bằng nhau, giữ dòng xuất hiện trước.
3. Tạo `Installs_10k` để đánh dấu cận dưới lượt cài đạt ít nhất 10.000. Chia giá niêm yết thành hai nhóm đến 2 USD và trên 2 USD.
4. Chỉ giữ danh mục có ít nhất 10 cặp ở mỗi nhóm giá. Tỷ lệ gộp chung và tỷ lệ theo danh mục dùng cùng tập này.

In [ ]:
# ---Thiết lập ngưỡng mốc lượt cài, mốc giá và cỡ nhóm tối thiểu---
Q21_INSTALL_TARGET = 10_000
Q21_PRICE_CUTOFF = 2.0
Q21_MIN_GROUP_SIZE = 10

# ---Lọc ứng dụng trả phí và kiểm tra các cột cần dùng---
q21_steps = []
rows_before = len(apps)
q21_paid = apps.loc[apps["Type"] == "Paid"].copy()
q21_steps.append({
    "Step": "Filter Paid apps (Type == 'Paid')",
    "Rows before": rows_before,
    "Removed": rows_before - len(q21_paid),
    "Rows after": len(q21_paid)
})

invalid_paid = (
    q21_paid["Price"].isna()
    | (q21_paid["Price"] <= 0)
    | q21_paid["Installs"].isna()
    | q21_paid["Category"].isna()
)
invalid_count = int(invalid_paid.sum())
print(f"Invalid or incomplete paid rows: {invalid_count}")
if invalid_count:
    raise ValueError("Paid rows contain invalid or missing values.")

# ---Chọn một bản ghi đại diện mỗi cặp ứng dụng và danh mục---
rows_before = len(q21_paid)
q21_paid = q21_paid.sort_values(
    "Last Updated", ascending=False, kind="stable"
)
q21_paid = q21_paid.sort_values(
    "Reviews", ascending=False, kind="stable"
)
q21_paid = q21_paid.drop_duplicates(
    subset=["App", "Category"],
    keep="first"
).copy()
q21_paid["Category"] = q21_paid["Category"].astype(str)
q21_steps.append({
    "Step": "Deduplicate to 1 row per (App, Category)",
    "Rows before": rows_before,
    "Removed": rows_before - len(q21_paid),
    "Rows after": len(q21_paid)
})

# ---Tạo biến chỉ báo đạt mốc và hai nhóm giá---
q21_paid["Installs_10k"] = q21_paid["Installs"] >= Q21_INSTALL_TARGET
q21_paid["Price_group"] = np.where(
    q21_paid["Price"] <= Q21_PRICE_CUTOFF,
    "<= $2",
    "> $2"
)

# ---Giữ danh mục có đủ quan sát ở cả hai khoảng giá---
cat_counts = (
    q21_paid.groupby(["Category", "Price_group"])
    .size()
    .unstack(fill_value=0)
)
has_enough = (
    (cat_counts["<= $2"] >= Q21_MIN_GROUP_SIZE)
    & (cat_counts["> $2"] >= Q21_MIN_GROUP_SIZE)
)
q21_eligible_cats = cat_counts.loc[has_enough].index.tolist()

rows_before = len(q21_paid)
q21_comp = q21_paid.loc[
    q21_paid["Category"].isin(q21_eligible_cats)
].copy()
q21_steps.append({
    "Step": f"Keep categories with >= {Q21_MIN_GROUP_SIZE} per band",
    "Rows before": rows_before,
    "Removed": rows_before - len(q21_comp),
    "Rows after": len(q21_comp)
})

q21_row_report = pd.DataFrame(q21_steps)
display(q21_row_report)
share_kept = len(q21_comp) / len(q21_paid) * 100
print(
    f"Eligible categories: {len(q21_eligible_cats)}\n"
    f"Pairs retained: {len(q21_comp):,} ({share_kept:.2f}%)"
)
display(cat_counts.loc[q21_eligible_cats])

**Nhận xét về mẫu so sánh**

- Lọc từ **10.357** dòng sạch còn **765** bản ghi trả phí. Cả 765 bản ghi có giá dương và đầy đủ các cột cần dùng.
- Loại **7** dòng lặp trong cùng danh mục, còn **758** cặp ứng dụng–danh mục. Một ứng dụng có thể xuất hiện ở hai danh mục vì mỗi cặp là một quan sát.
- **6/30 danh mục có ứng dụng trả phí** đạt ngưỡng tối thiểu ở cả hai nhóm giá. Tập so sánh còn **536 cặp**, chiếm **70,71%** trong 758 cặp.

#### **2. So sánh tỷ lệ đạt mốc chung và theo danh mục (D3, D4)**

- Với mỗi nhóm giá trong tập so sánh, lấy số cặp đạt cận dưới 10.000 lượt cài chia cho tổng số cặp. Tính tỷ lệ gộp và tỷ lệ riêng trong từng danh mục trên cùng tập 536 cặp. Bảng toàn bộ 758 cặp dùng làm mốc tham chiếu.
- Tóm tắt chênh lệch trong từng danh mục bằng trung bình có trọng số. Trọng số là số cặp của danh mục chia cho 536.
- Chia nhỏ hai nhóm giá thành năm khoảng để mô tả tỷ lệ ở từng khoảng. Mỗi khoảng được đọc cùng cỡ nhóm.

In [ ]:
# ---Tính tỷ lệ đạt mốc chung trên tập trả phí và tập so sánh---
price_order = ["<= $2", "> $2"]

def q21_summarize_group(df, group_col, order):
    rows = []
    for label in order:
        sub = df.loc[df[group_col] == label, "Installs_10k"]
        n_obs = len(sub)
        n_hit = int(sub.sum())
        rate = n_hit / n_obs * 100 if n_obs > 0 else np.nan
        rows.append({
            group_col: label,
            "Pairs (n)": n_obs,
            "Installs >= 10K (n)": n_hit,
            "Hit rate (%)": rate
        })
    return pd.DataFrame(rows)

q21_all_summary = q21_summarize_group(
    q21_paid, "Price_group", price_order
)
q21_comp_summary = q21_summarize_group(
    q21_comp, "Price_group", price_order
)

print(f"Reference: all paid pairs (n = {len(q21_paid):,})")
display(q21_all_summary.round(2))
print(f"Main comparison set (n = {len(q21_comp):,})")
display(q21_comp_summary.round(2))

rate_le2_comp = q21_comp_summary.loc[0, "Hit rate (%)"]
rate_gt2_comp = q21_comp_summary.loc[1, "Hit rate (%)"]
q21_overall_diff = rate_gt2_comp - rate_le2_comp

# ---Đối chiếu tỷ lệ trong từng danh mục thuộc tập so sánh---
cat_rows = []
for cat in q21_eligible_cats:
    sub = q21_comp.loc[q21_comp["Category"] == cat]
    g_le = sub.loc[sub["Price_group"] == "<= $2", "Installs_10k"]
    g_gt = sub.loc[sub["Price_group"] == "> $2", "Installs_10k"]
    r_le = g_le.mean() * 100
    r_gt = g_gt.mean() * 100
    cat_rows.append({
        "Category": cat,
        "n (<= $2)": len(g_le),
        "Hit (<= $2)": int(g_le.sum()),
        "Rate <= $2 (%)": r_le,
        "n (> $2)": len(g_gt),
        "Hit (> $2)": int(g_gt.sum()),
        "Rate > $2 (%)": r_gt,
        "Total n": len(sub),
        "Diff >$2 - <=$2 (pp)": r_gt - r_le
    })

q21_cat_table = pd.DataFrame(cat_rows).sort_values(
    by="Diff >$2 - <=$2 (pp)", ascending=False
).reset_index(drop=True)
display(q21_cat_table.round(2))

# ---Tính chênh lệch trung bình có trọng số theo quy mô danh mục---
weights = q21_cat_table["Total n"]
q21_weighted_diff = (
    (weights * q21_cat_table["Diff >$2 - <=$2 (pp)"]).sum()
    / weights.sum()
)
n_gt_higher = int((q21_cat_table["Diff >$2 - <=$2 (pp)"] > 0).sum())
print(
    f"Overall diff (> $2 minus <= $2): {q21_overall_diff:+.2f} pp"
)
print(
    f"Category-weighted diff:          {q21_weighted_diff:+.2f} pp"
)
print(
    f"Categories where > $2 is higher: "
    f"{n_gt_higher}/{len(q21_cat_table)}"
)

# ---Mô tả năm khoảng giá trong tập so sánh---
band_bins = [0, 1.0, 2.0, 5.0, 10.0, np.inf]
band_labels = ["<= 1", "(1, 2]", "(2, 5]", "(5, 10]", "> 10"]
q21_comp["Price_band"] = pd.cut(
    q21_comp["Price"], bins=band_bins, labels=band_labels
)
q21_band_table = q21_summarize_group(
    q21_comp, "Price_band", band_labels
)
print("\nPrice bands within the main comparison set")
display(q21_band_table.round(2))

**Nhận xét về tỷ lệ đạt mốc**

- **Tỷ lệ gộp trên 536 cặp.** Nhóm trên 2 USD đạt **47,27%** (147/311), nhóm đến 2 USD đạt **25,33%** (57/225). Chênh lệch là **21,93 điểm phần trăm**, tính từ tỷ lệ trước khi làm tròn. Trên toàn bộ 758 cặp trả phí, thứ tự cũng giống vậy với **42,53%** và **25,09%**.
- **So sánh theo danh mục.** Nhóm trên 2 USD có tỷ lệ cao hơn ở **6/6 danh mục**. Chênh lệch dao động từ **8,90 điểm phần trăm** ở `MEDICAL` đến **42,31** ở `COMMUNICATION`. Chênh lệch trung bình có trọng số theo cỡ danh mục là **23,93 điểm phần trăm**.
- **Theo khoảng giá chi tiết.** Trong mẫu so sánh, khoảng trên 2 đến 5 USD có tỷ lệ cao nhất (**51,58%**, 114/221), tiếp theo là trên 5 đến 10 USD (**43,18%**, 19/44) và trên 10 USD (**30,43%**, 14/46). Tỷ lệ quan sát được thay đổi theo từng khoảng giá.

#### **3. Trực quan hóa kết quả**

In [ ]:
# ---Biểu đồ 1: Tỷ lệ đạt 10K lượt cài chung và theo danh mục---
plot_labels = ["Overall\n(6 categories)"] + q21_cat_table[
    "Category"
].tolist()
rates_le2 = (
    [rate_le2_comp] + q21_cat_table["Rate <= $2 (%)"].tolist()
)
rates_gt2 = (
    [rate_gt2_comp] + q21_cat_table["Rate > $2 (%)"].tolist()
)
n_le2_list = [int(q21_comp_summary.loc[0, "Pairs (n)"])] + (
    q21_cat_table["n (<= $2)"].tolist()
)
n_gt2_list = [int(q21_comp_summary.loc[1, "Pairs (n)"])] + (
    q21_cat_table["n (> $2)"].tolist()
)

x_pos = np.arange(len(plot_labels))
bar_width = 0.36

fig, ax = plt.subplots(figsize=(11.5, 5.2))
bars_le2 = ax.bar(
    x_pos - bar_width / 2,
    rates_le2,
    width=bar_width,
    color="#4C78A8",
    label="Price <= 2 USD"
)
bars_gt2 = ax.bar(
    x_pos + bar_width / 2,
    rates_gt2,
    width=bar_width,
    color="#F58518",
    label="Price > 2 USD"
)

for bars, counts in [(bars_le2, n_le2_list), (bars_gt2, n_gt2_list)]:
    labels = []
    for bar, count in zip(bars, counts):
        rate = bar.get_height()
        labels.append(f"{rate:.1f}%\n(n={count})")
    ax.bar_label(bars, labels=labels, padding=3, fontsize=8)

ax.axvline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_xticks(x_pos)
ax.set_xticklabels(plot_labels, rotation=18, ha="right")
ax.set_ylim(0, 100)
ax.set_ylabel("Share with install floor >= 10,000 (%)")
ax.set_xlabel("Scope or category (n = app-category pairs)")
ax.set_title(
    "Paid Apps Reaching >= 10,000 Installs: "
    "Price <= 2 USD vs > 2 USD"
)
ax.grid(axis="y", linestyle=":", alpha=0.5)
ax.legend(title="Price group", loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# ---Biểu đồ 2: Chênh lệch theo danh mục và tỷ lệ theo khoảng giá---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# ---Biểu đồ trái: Chênh lệch tỷ lệ theo từng danh mục---
cat_plot = q21_cat_table.sort_values(
    by="Diff >$2 - <=$2 (pp)", ascending=True
)
y_pos = np.arange(len(cat_plot))
y_labels = [
    f"{category}\n(n={low}/{high})"
    for category, low, high in zip(
        cat_plot["Category"], cat_plot["n (<= $2)"],
        cat_plot["n (> $2)"]
    )
]

gaps = cat_plot["Diff >$2 - <=$2 (pp)"]
gap_bars = axes[0].barh(y_pos, gaps, color="#F58518", alpha=0.88)
gap_labels = [f"{gap:+.1f} pp" for gap in gaps]
axes[0].bar_label(
    gap_bars, labels=gap_labels, padding=4, fontsize=8.5,
    bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.6}
)
axes[0].axvline(0, color="black", linewidth=1)
axes[0].axvline(
    q21_overall_diff,
    color="#4C78A8",
    linestyle="--",
    linewidth=1.4,
    label=f"Overall ({q21_overall_diff:+.1f} pp)"
)
axes[0].axvline(
    q21_weighted_diff,
    color="#2CA02C",
    linestyle=":",
    linewidth=1.8,
    label=f"Weighted avg ({q21_weighted_diff:+.1f} pp)"
)

axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(y_labels, fontsize=8.5)
axes[0].margins(x=0.2)
axes[0].set_xlabel(
    "Rate difference: (> 2 USD) minus (<= 2 USD) (pp)"
)
axes[0].set_title("Within-Category Gap (n: <= 2 USD / > 2 USD)")
axes[0].grid(axis="x", linestyle=":", alpha=0.5)
axes[0].legend(loc="lower right", fontsize=8.5)

# ---Biểu đồ phải: Tỷ lệ đạt 10K lượt cài theo 5 khoảng giá---
band_x = np.arange(len(q21_band_table))
band_colors = ["#4C78A8"] * 2 + ["#F58518"] * 3
band_bars = axes[1].bar(
    band_x,
    q21_band_table["Hit rate (%)"],
    color=band_colors,
    alpha=0.9
)
axes[1].axvline(
    1.5,
    color="black",
    linestyle="--",
    linewidth=1.1,
    label="2 USD cutoff"
)

band_text = []
for rate, count in zip(
    q21_band_table["Hit rate (%)"], q21_band_table["Pairs (n)"]
):
    band_text.append(f"{rate:.1f}%\n(n={count})")
axes[1].bar_label(
    band_bars, labels=band_text, padding=3, fontsize=8.5
)

axes[1].set_xticks(band_x)
axes[1].set_xticklabels(q21_band_table["Price_band"])
axes[1].set_ylim(0, 100)
axes[1].set_xlabel("Listed price band (USD)")
axes[1].set_ylabel("Share with install floor >= 10,000 (%)")
axes[1].set_title(
    "Hit Rate Across Finer Price Bands (Comparable Set)"
)
axes[1].grid(axis="y", linestyle=":", alpha=0.5)
axes[1].legend(loc="upper right", fontsize=8.5)

plt.tight_layout()
plt.show()

#### **4. Trả lời câu hỏi và giới hạn**

**Kết luận**

Trên tập 536 cặp ứng dụng–danh mục, nhóm giá trên 2 USD đạt mốc 10.000 lượt cài thường xuyên hơn nhóm đến 2 USD (**47,27% so với 25,33%**). Thứ tự này xuất hiện trong cả sáu danh mục đủ điều kiện so sánh.

**Ý nghĩa thực tế**

Khi đã chọn mô hình trả phí lúc tải, nhà phát triển có thể dùng tỷ lệ trong danh mục của mình làm mốc tham khảo. Chẳng hạn, ở `GAME`, hai nhóm giá lần lượt đạt **76,19%** và **40,00%**. Tỷ lệ toàn mẫu và tỷ lệ từng danh mục giúp đặt kỳ vọng ở hai phạm vi khác nhau.

**Giới hạn**

- Giá là mức niêm yết tại thời điểm thu thập, còn lượt cài được ghi theo cận dưới tích lũy. Ước lượng tác động của một lần đổi giá cần lịch sử giá và lượt cài theo thời gian.
- Kết quả chính thuộc sáu danh mục đủ cỡ nhóm. `COMMUNICATION` chỉ có 13 và 14 cặp ở hai khoảng giá, nên mức chênh cụ thể tại đây cần được đọc cùng cỡ mẫu.

### **2.2. Lượt cài và điểm đánh giá ở ứng dụng miễn phí**

**Người thực hiện:** Phạm Hoàng Tuấn Anh

**Câu hỏi**

Trong các ứng dụng miễn phí có ít nhất 100 lượt đánh giá, ứng dụng có cận dưới lượt cài cao hơn có xu hướng được chấm điểm cao hơn không, và mối liên hệ mạnh đến đâu? Kết quả thay đổi thế nào khi dùng ngưỡng 50 hoặc 200 lượt đánh giá?

**Động lực**

Sau khi phát hành, đội phát triển theo dõi lượt cài để biết mức tiếp cận và điểm đánh giá để đọc phản hồi người dùng. Kết quả giúp họ hiểu hai chỉ số thường đi cùng nhau đến mức nào khi đối chiếu với các ứng dụng miễn phí khác.

**Các tiêu chí chiều sâu**

- **D5** — Ước lượng tương quan Spearman giữa cận dưới lượt cài (`Installs`) và điểm đánh giá (`Rating`), kèm khoảng tin cậy bootstrap 95%. Dùng một bản ghi đại diện mỗi tên ứng dụng trước khi lấy mẫu lại.
- **D6** — Tính lại tương quan và khoảng tin cậy với ngưỡng tối thiểu 50 và 200 lượt đánh giá. Báo cỡ mẫu và đối chiếu chiều, độ lớn của tương quan qua ba ngưỡng.

**Phạm vi diễn giải**

Kết quả mô tả mối liên hệ trong mẫu ứng dụng miễn phí có điểm và đủ lượt đánh giá, tại thời điểm thu thập. `Installs` ghi cận dưới của khoảng cài đặt. Cơ cấu danh mục và độ phủ của mẫu có thể ảnh hưởng đến tương quan.

#### **1. Chuẩn bị mẫu theo số lượt đánh giá**

1. Giữ các bản ghi `Type == "Free"`.
2. Giữ một dòng cho mỗi tên ứng dụng. Ưu tiên số lượt đánh giá cao hơn, rồi ngày cập nhật mới hơn. Nếu cả hai bằng nhau, giữ dòng xuất hiện trước. Mỗi tên đóng góp một quan sát để tính tương quan và lấy mẫu bootstrap.
3. Kiểm tra số dòng thiếu `Rating` tại mỗi ngưỡng. Giữ các ứng dụng có đủ `Rating` và `Installs`.
4. Từ tập còn lại, lọc riêng ba mẫu `Reviews >= 50`, `>= 100` và `>= 200`. Mẫu chính dùng ngưỡng 100. Ba mẫu dùng chung một tập đầu vào và có thể chứa các ứng dụng giống nhau.

In [ ]:
# ---Thiết lập ngưỡng chính và các ngưỡng kiểm tra độ bền---
Q22_MAIN_THRESHOLD = 100
Q22_THRESHOLDS = [50, 100, 200]

# ---Lọc các bản ghi ứng dụng miễn phí---
q22_steps = []
rows_before = len(apps)
q22_free = apps.loc[apps["Type"] == "Free"].copy()
q22_steps.append({
    "Step": "Filter Free apps (Type == 'Free')",
    "Rows before": rows_before,
    "Removed": rows_before - len(q22_free),
    "Rows after": len(q22_free)
})

# ---Chọn một bản ghi đại diện mỗi tên ứng dụng---
rows_before = len(q22_free)
q22_free = q22_free.sort_values(
    "Last Updated", ascending=False, kind="stable"
)
q22_free = q22_free.sort_values(
    "Reviews", ascending=False, kind="stable"
)
q22_free = q22_free.drop_duplicates(
    subset=["App"], keep="first"
).copy()
q22_steps.append({
    "Step": "Deduplicate to 1 representative row per App",
    "Rows before": rows_before,
    "Removed": rows_before - len(q22_free),
    "Rows after": len(q22_free)
})

# ---Đếm số ứng dụng thiếu điểm ở từng ngưỡng lượt đánh giá---
missing_rating = q22_free["Rating"].isna()
for threshold in Q22_THRESHOLDS:
    missing_at_threshold = int(
        (missing_rating & q22_free["Reviews"].ge(threshold)).sum()
    )
    print(
        f"Missing Rating at Reviews >= {threshold}: "
        f"{missing_at_threshold}"
    )

# ---Giữ các ứng dụng có điểm và cận dưới lượt cài---
rows_before = len(q22_free)
q22_rated = q22_free.dropna(subset=["Rating", "Installs"]).copy()
q22_steps.append({
    "Step": "Keep apps with Rating and Installs",
    "Rows before": rows_before,
    "Removed": rows_before - len(q22_rated),
    "Rows after": len(q22_rated)
})

# ---Tạo từng mẫu từ cùng tập ứng dụng đủ dữ liệu---
q22_samples = {}
for threshold in Q22_THRESHOLDS:
    sample = q22_rated.loc[q22_rated["Reviews"].ge(threshold)].copy()
    q22_samples[threshold] = sample
    q22_steps.append({
        "Step": f"Filter rated free apps by Reviews >= {threshold}",
        "Rows before": len(q22_rated),
        "Removed": len(q22_rated) - len(sample),
        "Rows after": len(sample)
    })

q22_row_report = pd.DataFrame(q22_steps)
display(q22_row_report)
q22_main = q22_samples[Q22_MAIN_THRESHOLD]

**Nhận xét về cỡ mẫu**

- Từ **10.357** bản ghi sạch còn **9.592** dòng miễn phí. Giữ một dòng mỗi tên loại **687** dòng, còn **8.905** tên ứng dụng.
- Sau khi loại **1.311** dòng thiếu điểm, còn **7.594** ứng dụng có đủ `Rating` và `Installs`. Trong bước kiểm tra trước khi lọc điểm, có **60**, **32** và **17** ứng dụng thiếu `Rating` dù lần lượt đạt ngưỡng 50, 100 và 200 lượt đánh giá.
- Mẫu chính có **5.934** ứng dụng. Hai mẫu ở ngưỡng 50 và 200 có lần lượt **6.305** và **5.536** ứng dụng.

#### **2. Ước lượng tương quan và kiểm tra độ bền (D5, D6)**

- **D5.** Dùng tương quan hạng Spearman để đo hướng và độ mạnh của xu hướng đơn điệu. `Installs` gồm các mốc rời rạc và `Rating` nằm trong thang 1–5. Lấy mẫu lại có hoàn lại theo ứng dụng **1.000 lần**, dùng hạt giống 42 và lấy hai phân vị 2,5% và 97,5% làm khoảng tin cậy.
- **D6.** Lặp lại quy trình ở ngưỡng 50 và 200 lượt đánh giá, rồi đối chiếu hệ số với mẫu chính ở ngưỡng 100.
- Mô tả thêm phân bố điểm theo các mốc `Installs` có sẵn trong mẫu chính. Gộp các mốc nhỏ nhất và lớn nhất để dễ đọc biểu đồ.

In [ ]:
# ---Hàm ước lượng Spearman và khoảng tin cậy bootstrap 95%---
Q22_N_BOOT = 1000
Q22_SEED = 42

def q22_spearman_bootstrap(df, n_boot=Q22_N_BOOT, seed=Q22_SEED):
    install_floors = df["Installs"].to_numpy(dtype="float64")
    ratings = df["Rating"].to_numpy(dtype="float64")
    rho, p_value = spearmanr(install_floors, ratings)

    rng = np.random.default_rng(seed)
    n_obs = len(df)
    boot_rhos = np.empty(n_boot, dtype="float64")
    for draw in range(n_boot):
        indices = rng.integers(0, n_obs, size=n_obs)
        boot_rhos[draw], _ = spearmanr(
            install_floors[indices], ratings[indices]
        )

    ci_low, ci_high = np.percentile(boot_rhos, [2.5, 97.5])
    return {
        "rho": float(rho),
        "p_value": float(p_value),
        "ci_low": float(ci_low),
        "ci_high": float(ci_high),
        "boot_rhos": boot_rhos
    }

# ---Tính kết quả qua ba ngưỡng lượt đánh giá---
q22_results = []
q22_boot_store = {}
for threshold in Q22_THRESHOLDS:
    sample = q22_samples[threshold]
    result = q22_spearman_bootstrap(sample)
    q22_boot_store[threshold] = result["boot_rhos"]
    q22_results.append({
        "Threshold": f"Reviews >= {threshold}",
        "Apps (n)": len(sample),
        "Spearman rho": result["rho"],
        "CI 95% low": result["ci_low"],
        "CI 95% high": result["ci_high"],
        "CI width": result["ci_high"] - result["ci_low"],
        "p-value": result["p_value"]
    })

q22_summary = pd.DataFrame(q22_results)
display(q22_summary.round({
    "Spearman rho": 4,
    "CI 95% low": 4,
    "CI 95% high": 4,
    "CI width": 4
}))

# ---Thống kê điểm đánh giá theo bậc lượt cài (mẫu chính)---
tier_bins = [
    0, 5_000, 10_000, 50_000, 100_000, 500_000,
    1_000_000, 5_000_000, 10_000_000, 50_000_000, np.inf
]
tier_labels = [
    "<=5K", "10K", "50K", "100K", "500K",
    "1M", "5M", "10M", "50M", ">50M"
]
q22_main["Install_tier"] = pd.cut(
    q22_main["Installs"], bins=tier_bins, labels=tier_labels,
    include_lowest=True
)

tier_rows = []
for tier in tier_labels:
    vals = q22_main.loc[q22_main["Install_tier"] == tier, "Rating"]
    q25 = vals.quantile(0.25)
    q75 = vals.quantile(0.75)
    tier_rows.append({
        "Install tier": tier,
        "Apps (n)": len(vals),
        "Mean": vals.mean(),
        "Median": vals.median(),
        "Std": vals.std(),
        "Q25": q25,
        "Q75": q75,
        "IQR": q75 - q25
    })

q22_tier_table = pd.DataFrame(tier_rows)
print(
    "Rating by install floor "
    f"(main sample: Reviews >= {Q22_MAIN_THRESHOLD})"
)
display(q22_tier_table.round(3))

**Nhận xét về kết quả**

- **Mẫu chính (5.934 ứng dụng).** Spearman là **0,1626**, với khoảng tin cậy bootstrap 95% **[0,1365; 0,1876]**. Ước lượng cho thấy liên hệ dương, mức yếu.
- **Phân bố theo mốc lượt cài.** Điểm trung bình giảm từ **4,069** ở mốc 10 nghìn xuống **4,024** ở mốc 50 nghìn. Sau đó, điểm trung bình tăng dần đến **4,370** ở nhóm trên 50 triệu. Khoảng tứ phân vị của điểm thu hẹp từ **0,70** ở mốc 10 nghìn xuống **0,20** ở nhóm trên 50 triệu.
- **Nhóm cài đặt thấp nhất.** Nhóm tối đa 5 nghìn có **87 ứng dụng**, với trung vị điểm **4,50**, cao hơn trung vị các nhóm còn lại. Nhận xét này chỉ áp dụng cho 87 ứng dụng đã qua ngưỡng ít nhất 100 lượt đánh giá.
- **Kiểm tra độ bền.** Spearman là **0,1606** ở ngưỡng 50 và **0,1576** ở ngưỡng 200. Chênh lệch lớn nhất giữa ba ước lượng khoảng **0,005**. Kết luận về chiều dương và độ mạnh yếu giữ được trong ba mẫu đã xét.

#### **3. Trực quan hóa kết quả**

In [ ]:
# ---Biểu đồ 1: Điểm đánh giá theo bậc lượt cài (mẫu chính)---
box_data = []
for tier in tier_labels:
    vals = q22_main.loc[q22_main["Install_tier"] == tier, "Rating"]
    box_data.append(vals.to_numpy())
x_ticks = np.arange(1, len(tier_labels) + 1)
tick_text = [
    f"{tier}\n(n={count})"
    for tier, count in zip(
        q22_tier_table["Install tier"], q22_tier_table["Apps (n)"]
    )
]

fig, ax = plt.subplots(figsize=(12, 5.2))
bp = ax.boxplot(
    box_data,
    positions=x_ticks,
    widths=0.55,
    patch_artist=True,
    flierprops={
        "markersize": 2,
        "markeredgecolor": "#667085",
        "alpha": 0.35
    },
    medianprops={"color": "#1F2937", "linewidth": 1.8}
)
for patch in bp["boxes"]:
    patch.set_facecolor("#89B0D9")
    patch.set_alpha(0.75)

ax.plot(
    x_ticks,
    q22_tier_table["Mean"],
    color="#D62728",
    marker="o",
    linewidth=1.8,
    label="Group mean rating"
)
ax.plot(
    x_ticks,
    q22_tier_table["Median"],
    color="#1F2937",
    linestyle="--",
    marker="s",
    markersize=4,
    linewidth=1.2,
    label="Group median rating"
)

ax.set_xticks(x_ticks)
ax.set_xticklabels(tick_text, fontsize=8.5)
ax.set_ylim(1, 5.15)
ax.set_xlabel("Installs lower-bound tier (with sample size n)")
ax.set_ylabel("Rating (1 to 5 stars)")
ax.set_title(
    "Rating Distribution by Installs Tier "
    f"(Free apps, Reviews >= {Q22_MAIN_THRESHOLD}, "
    f"n = {len(q22_main):,})"
)
ax.grid(axis="y", linestyle=":", alpha=0.5)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# ---Biểu đồ 2: Spearman và phân bố bootstrap qua ba ngưỡng---
fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8))

# ---Biểu đồ trái: Ước lượng điểm và khoảng tin cậy bootstrap 95%---
y_pos = np.arange(len(q22_summary))
rhos = q22_summary["Spearman rho"].to_numpy()
err_low = rhos - q22_summary["CI 95% low"].to_numpy()
err_high = q22_summary["CI 95% high"].to_numpy() - rhos
y_labels = [
    f"{label}\n(n={count:,})"
    for label, count in zip(
        q22_summary["Threshold"], q22_summary["Apps (n)"]
    )
]

axes[0].errorbar(
    rhos,
    y_pos,
    xerr=[err_low, err_high],
    fmt="o",
    color="#2B6CB0",
    ecolor="#2B6CB0",
    elinewidth=2,
    capsize=5,
    label="Spearman rho (95% bootstrap CI)"
)
axes[0].axvline(
    0,
    color="gray",
    linestyle="--",
    linewidth=1,
    label="No correlation"
)

for position, estimate, low, high in zip(
    y_pos, rhos, q22_summary["CI 95% low"],
    q22_summary["CI 95% high"]
):
    label_txt = f"{estimate:.4f} [{low:.4f}, {high:.4f}]"
    axes[0].text(
        estimate,
        position + 0.16,
        label_txt,
        ha="center",
        va="bottom",
        fontsize=8.5
    )

axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(y_labels, fontsize=9)
axes[0].set_ylim(-0.5, len(q22_summary) - 0.35)
upper_limit = q22_summary["CI 95% high"].max()
axes[0].set_xlim(-0.02, upper_limit + 0.06)
axes[0].set_xlabel("Spearman rank correlation (rho)")
axes[0].set_title("Spearman Correlation Across Review Thresholds")
axes[0].grid(axis="x", linestyle=":", alpha=0.5)
axes[0].legend(loc="lower left", fontsize=8.5)

# ---Biểu đồ phải: Phân bố bootstrap của Spearman qua ba ngưỡng---
colors = ["#4C78A8", "#F58518", "#2CA02C"]
all_draws = np.concatenate(list(q22_boot_store.values()))
bin_edges = np.histogram_bin_edges(all_draws, bins=28)
for threshold, color in zip(Q22_THRESHOLDS, colors):
    sample_size = len(q22_samples[threshold])
    axes[1].hist(
        q22_boot_store[threshold],
        bins=bin_edges,
        density=True,
        histtype="step",
        linewidth=2,
        color=color,
        label=f"Reviews >= {threshold} (n={sample_size:,})"
    )

axes[1].set_xlabel(f"Bootstrap Spearman rho ({Q22_N_BOOT:,} draws)")
axes[1].set_ylabel("Density")
axes[1].set_title("Bootstrap Distributions Across Thresholds")
axes[1].grid(axis="y", linestyle=":", alpha=0.5)
axes[1].legend(loc="upper right", fontsize=8.5)

plt.tight_layout()
plt.show()

#### **4. Trả lời câu hỏi và giới hạn**

**Kết luận**

Trong **5.934 ứng dụng miễn phí** có ít nhất 100 lượt đánh giá, ứng dụng có cận dưới lượt cài cao hơn có xu hướng được chấm điểm cao hơn. Liên hệ ở mức yếu với Spearman **0,1626**. Kết luận về chiều và độ mạnh tương tự ở hai ngưỡng 50 và 200 lượt đánh giá.

**Ý nghĩa thực tế**

Đội phát triển nên đọc lượt cài và điểm sao cùng nhau khi rà soát ứng dụng sau phát hành. Lượt cài phản ánh mức tiếp cận. Điểm sao phản ánh đánh giá của những người đã chấm điểm. Trong mẫu này, hai chỉ số chỉ đi cùng nhau ở mức yếu, và điểm giữa các bậc lượt cài vẫn chồng lấn.

**Giới hạn**

- `Installs` ghi cận dưới theo từng mốc. `Rating` phản ánh điểm ghi nhận tại thời điểm thu thập năm 2018.
- Tương quan gộp có thể chịu ảnh hưởng của cơ cấu danh mục và thời gian ứng dụng đã có mặt trên cửa hàng. Dữ liệu cắt ngang giới hạn khả năng xác định chiều tác động.
- Kết quả áp dụng cho ứng dụng miễn phí có đủ điểm và lượt đánh giá trong mẫu. Ba mẫu theo ngưỡng có chung nhiều ứng dụng, nên phép kiểm tra độ bền phản ánh độ nhạy với quy tắc lọc.

### **2.3. Độ mới của bản cập nhật và mức phổ biến của ứng dụng**

**Người thực hiện:** *(điền tên thành viên)*

**Câu hỏi:** Ứng dụng được cập nhật gần đây có tỷ lệ đạt từ 1.000.000 lượt cài trở lên cao hơn ứng dụng lâu không cập nhật hay không? Mối liên hệ này còn giữ khi so sánh trong cùng danh mục và cùng loại miễn phí hoặc trả phí không?

**Động lực:** Khám phá cho thấy ứng dụng khác nhau rất nhiều về `Installs`, `Price` và danh mục, trong khi `Last Updated` chưa được khai thác. Kết quả giúp nhà phát triển và nhà phát hành cân nhắc việc duy trì cập nhật. Đây là mối liên hệ quan sát: ứng dụng phổ biến có thể được cập nhật thường xuyên hơn, chứ chưa chắc cập nhật làm tăng lượt cài.

**Các tiêu chí chiều sâu:**

- **D2:** Tính thời gian từ lần cập nhật cuối đến ngày tham chiếu, so sánh theo nhóm thời gian và theo tháng cập nhật.
- **D3:** Tạo `Days_since_update`, nhóm thời gian, nhóm kích thước và nhãn phổ biến.
- **D4:** So sánh trong từng tầng danh mục × loại (Free hoặc Paid), rồi tổng hợp bằng trọng số cỡ tầng và kiểm tra riêng từng danh mục.
- **D5:** Khoảng tin cậy bootstrap 95% (lấy mẫu lại ứng dụng) và hệ số Spearman kèm p-value.
- **D6:** Đổi ngưỡng "mới", cách phân tầng, phạm vi mẫu và ngưỡng phổ biến.

#### **1. Chuẩn bị dữ liệu và tạo biến (D2, D3)**

1. Từ bảng `apps` đã làm sạch, nhóm giữ một dòng cho mỗi tên ứng dụng (ưu tiên `Reviews` lớn nhất, rồi `Last Updated` mới hơn) để mỗi ứng dụng chỉ được tính một lần.
2. Ngày tham chiếu là `Last Updated` muộn nhất trong dữ liệu (xấp xỉ ngày thu thập). `Days_since_update` là số ngày từ lần cập nhật cuối đến ngày này.
3. `Age_group` chia thành 5 nhóm; `Popular` bằng 1 nếu cận dưới `Installs` từ 1.000.000; `Size_group` chia kích thước thành 4 nhóm (gồm "Varies").

In [ ]:
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

# ---------- 1. Một bản ghi cho mỗi tên ứng dụng ----------
q2_rows_before = len(apps)
q2 = (
    apps.sort_values(["Reviews", "Last Updated"], ascending=False, kind="stable")
    .drop_duplicates("App")
    .copy()
)

# ---------- 2. Biến thời gian và biến dẫn xuất ----------
# Ngày cập nhật muộn nhất trong dữ liệu được coi là ngày thu thập.
REF_DATE = q2["Last Updated"].max()
q2["Days_since_update"] = (REF_DATE - q2["Last Updated"]).dt.days
AGE_LABELS = ["≤30 ngày", "31–90 ngày", "91–180 ngày", "181–365 ngày", ">365 ngày"]
q2["Age_group"] = pd.cut(q2["Days_since_update"], [-1, 30, 90, 180, 365, np.inf], labels=AGE_LABELS)
q2["Popular"] = q2["Installs"].ge(1_000_000)
q2["Size_group"] = pd.cut(q2["Size_MB"], [0, 5, 20, np.inf], labels=["<5MB", "5–20MB", ">20MB"]).cat.add_categories("Varies").fillna("Varies").astype(str)
q2["Category"] = q2["Category"].astype(str)
q2["Type"] = q2["Type"].astype(str)

display(pd.Series({
    "Số dòng trước": q2_rows_before,
    "Số ứng dụng (một dòng mỗi tên)": len(q2),
    "Ngày tham chiếu": REF_DATE.date(),
    "Số ngày cập nhật: trung vị": int(q2["Days_since_update"].median()),
}, name="Giá trị"))

q2_age = q2.groupby("Age_group", observed=True).agg(
    Apps=("App", "size"),
    Popular_share=("Popular", "mean"),
    Paid_share=("Type", lambda s: s.eq("Paid").mean()),
    Median_size_MB=("Size_MB", "median"),
)
display(q2_age.round(3))

Từ 10.357 dòng, nhóm còn **9.659** ứng dụng. Ngày tham chiếu là 08/08/2018 và trung vị là 96 ngày kể từ lần cập nhật cuối. Tỷ lệ phổ biến giảm theo thời gian: **56,7%** ở nhóm ≤30 ngày xuống **18,5%** ở nhóm >365 ngày. Nhóm cũ có nhiều ứng dụng trả phí hơn (14,4% so với 3,7%) và kích thước nhỏ hơn (trung vị 6,4MB so với 19MB), nên cần kiểm soát các yếu tố này.

#### **2. Ước lượng chênh lệch và độ bất định (D4, D5)**

Nhóm gọi ứng dụng cập nhật trong 90 ngày gần nhất là "mới", còn lại là "cũ". Trong mỗi tầng danh mục × loại, nhóm tính chênh lệch tỷ lệ phổ biến (mới trừ cũ), chỉ giữ tầng có ít nhất 10 ứng dụng ở cả hai nhóm, rồi lấy trung bình với trọng số bằng cỡ tầng. Khoảng tin cậy 95% lấy bằng bootstrap 1.000 lần. Nhóm cũng tính Spearman giữa `Days_since_update` và `Installs`. Ô code có một phép kiểm tra nhỏ (`assert`) cho hàm tính chênh lệch có trọng số.

In [ ]:
def q2_adj_diff(codes, recent, pop, min_n=10):
    """Chênh lệch tỷ lệ phổ biến (mới − cũ) trung bình theo tầng, trọng số bằng cỡ tầng.
    Chỉ dùng tầng có ít nhất min_n ứng dụng ở cả hai nhóm."""
    k = codes.max() + 1
    n1 = np.bincount(codes[recent], minlength=k)
    n0 = np.bincount(codes[~recent], minlength=k)
    s1 = np.bincount(codes[recent], weights=pop[recent], minlength=k)
    s0 = np.bincount(codes[~recent], weights=pop[~recent], minlength=k)
    ok = (n1 >= min_n) & (n0 >= min_n)
    w = (n1 + n0)[ok]
    return np.sum(w * (s1[ok] / n1[ok] - s0[ok] / n0[ok])) / w.sum(), int(w.sum()), int(ok.sum())

# Kiểm tra nhanh: tầng A chênh 0,6 (20 app), tầng B chênh 0 (20 app), tầng C bị loại vì nhóm quá nhỏ.
_codes = np.array([0]*20 + [1]*20 + [2]*3)
_recent = np.array([True]*10 + [False]*10 + [True]*10 + [False]*10 + [True]*3)
_pop = np.array([1]*8 + [0]*2 + [1]*2 + [0]*8 + [1]*5 + [0]*5 + [1]*5 + [0]*5 + [1]*3, dtype=float)
_est, _n, _k = q2_adj_diff(_codes, _recent, _pop)
assert abs(_est - 0.3) < 1e-9 and _n == 40 and _k == 2

def q2_estimate(d, cutoff=90, strata=("Category", "Type"), thr=1_000_000, n_boot=1000, seed=42):
    """Ước lượng chênh lệch thô và sau phân tầng kèm CI bootstrap 95% (lấy mẫu lại ứng dụng)."""
    codes = d.groupby(list(strata)).ngroup().to_numpy()
    recent = d["Days_since_update"].le(cutoff).to_numpy()
    pop = d["Installs"].ge(thr).to_numpy(float)
    raw = lambda i: pop[i][recent[i]].mean() - pop[i][~recent[i]].mean()
    adj = lambda i: q2_adj_diff(codes[i], recent[i], pop[i])[0]
    all_i = np.arange(len(d))
    rng = np.random.default_rng(seed)
    boots = [rng.integers(0, len(d), len(d)) for _ in range(n_boot)]
    a, n_used, n_strata = q2_adj_diff(codes, recent, pop)
    return {"n": len(d), "Mới (n)": int(recent.sum()), "Cũ (n)": int((~recent).sum()),
            "Share mới": pop[recent].mean(), "Share cũ": pop[~recent].mean(),
            "Chênh thô": raw(all_i), "CI thô": tuple(np.percentile([raw(i) for i in boots], [2.5, 97.5])),
            "Chênh sau phân tầng": a, "CI phân tầng": tuple(np.percentile([adj(i) for i in boots], [2.5, 97.5])),
            "App dùng": n_used, "Số tầng": n_strata}

q2_main = q2_estimate(q2)
display(pd.Series(q2_main, name="Giá trị (mới = cập nhật ≤90 ngày)").to_frame())

rho, p = spearmanr(q2["Days_since_update"], q2["Installs"])
print(f"Spearman(Days_since_update, Installs) = {rho:.3f}, p = {p:.2g}")

- Tỷ lệ phổ biến là **49,9%** ở nhóm mới và **20,8%** ở nhóm cũ. Chênh lệch thô là **+0,291** (CI [0,273; 0,308]).
- Sau khi phân tầng danh mục × loại (38 tầng, 9.412 ứng dụng được dùng), chênh lệch còn **+0,281** (CI [0,262; 0,297]). Phân tầng gần như không làm mất mối liên hệ.
- Spearman giữa `Days_since_update` và `Installs` là **−0,332** (p < 0,001): cập nhật càng lâu thì `Installs` càng thấp, ở mức vừa phải.

#### **3. Trực quan hóa theo thời gian (D2)**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Biểu đồ 1: tỷ lệ phổ biến theo nhóm thời gian từ lần cập nhật cuối, CI bootstrap 95%
rng = np.random.default_rng(42)
shares, errs = [], []
for lab in AGE_LABELS:
    v = q2.loc[q2["Age_group"].eq(lab), "Popular"].to_numpy(float)
    m = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(1000)]
    lo, hi = np.percentile(m, [2.5, 97.5])
    shares.append(v.mean()); errs.append([v.mean() - lo, hi - v.mean()])
axes[0].bar(AGE_LABELS, shares, yerr=np.array(errs).T, capsize=4, color="C0")
axes[0].set(title="Tỷ lệ app ≥ 1 triệu lượt cài theo thời gian từ lần cập nhật cuối",
            xlabel="Thời gian từ lần cập nhật cuối đến ngày tham chiếu", ylabel="Tỷ lệ Installs ≥ 1.000.000 (CI 95%)")
axes[0].tick_params(axis="x", rotation=20)

# Biểu đồ 2: theo tháng cập nhật cuối (12 tháng gần nhất)
month = q2["Last Updated"].dt.to_period("M")
recent_months = month[month >= month.max() - 11].sort_values().unique()
cohort = q2[month.isin(recent_months)].groupby(month).agg(Popular_share=("Popular", "mean"), n=("App", "size"))
axes[1].plot(cohort.index.astype(str), cohort["Popular_share"], marker="o")
for x, (s, n) in enumerate(zip(cohort["Popular_share"], cohort["n"])):
    axes[1].annotate(f"n={n}", (x, s), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=7)
axes[1].set(title="Tỷ lệ app phổ biến theo tháng cập nhật cuối", xlabel="Tháng cập nhật cuối", ylabel="Tỷ lệ Installs ≥ 1.000.000")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

- Biểu đồ trái: tỷ lệ phổ biến giảm dần qua bốn nhóm đầu (56,7%, 38,9%, 23,5%, 22,9%) rồi gần như phẳng ở nhóm >365 ngày (18,5%); các CI không chồng lấn ở ba bước đầu.
- Biểu đồ phải: tỷ lệ phổ biến tăng mạnh theo tháng cập nhật cuối, từ khoảng 16–28% trong 2017 và đầu 2018 lên 35,1% (tháng 5), 39,6% (tháng 6), 50,2% (tháng 7) và 67,4% (tháng 8/2018, 984 ứng dụng). Mối liên hệ không thay đổi mạnh ở các tháng cũ hơn, nên tác động tập trung ở nhóm cập nhật rất gần.

#### **4. Kiểm soát theo danh mục (D4)**

Nhóm lặp lại ước lượng bên trong từng danh mục có ít nhất 150 ứng dụng, với tầng là `Type`. Mỗi danh mục dùng bootstrap 500 lần.

In [ ]:
# D4: kiểm tra trong từng danh mục (tầng theo Type), chỉ giữ danh mục đủ lớn.
cat_rows = []
for cat, g in q2.groupby("Category"):
    if len(g) < 150:
        continue
    r = q2_estimate(g, strata=("Type",), n_boot=500)
    cat_rows.append({"Category": cat, "n": r["n"], "Chênh sau phân tầng": r["Chênh sau phân tầng"], "CI": r["CI phân tầng"], "App dùng": r["App dùng"]})
q2_cat = pd.DataFrame(cat_rows).sort_values("Chênh sau phân tầng").reset_index(drop=True)
q2_cat_view = q2_cat.assign(CI=q2_cat["CI"].map(lambda v: f"[{v[0]:.3f}; {v[1]:.3f}]")).round(3)
display(q2_cat_view)

err = np.array([[r["Chênh sau phân tầng"] - r["CI"][0], r["CI"][1] - r["Chênh sau phân tầng"]] for _, r in q2_cat.iterrows()]).T
fig, ax = plt.subplots(figsize=(8, 0.4 * len(q2_cat) + 2))
ax.errorbar(q2_cat["Chênh sau phân tầng"], range(len(q2_cat)), xerr=err, fmt="o", capsize=3)
ax.axvline(0, color="gray", ls="--", lw=1)
ax.axvline(q2_main["Chênh sau phân tầng"], color="C3", ls=":", lw=1, label="Ước lượng chung")
ax.set_yticks(range(len(q2_cat)), [f"{c} (n={n})" for c, n in zip(q2_cat["Category"], q2_cat["n"])])
ax.set(title="Chênh lệch tỷ lệ phổ biến (cập nhật ≤90 ngày so với cũ hơn) theo danh mục",
       xlabel="Chênh lệch tỷ lệ Installs ≥ 1.000.000 (CI bootstrap 95%)")
ax.legend()
plt.tight_layout()
plt.show()

- Chênh lệch dương ở cả 20 danh mục, và CI của mọi danh mục đều nằm hẳn phía dương. Mối liên hệ không phải do một vài danh mục tạo ra.
- Độ lớn khác nhau: thấp nhất ở `MEDICAL` (+0,081, CI [0,047; 0,117]) và `DATING` (+0,175); cao nhất ở `SHOPPING` (+0,656), `COMMUNICATION` (+0,508) và `TRAVEL_AND_LOCAL` (+0,490). `GAME` (+0,285) và `FAMILY` (+0,222) gần mức chung.
- Các danh mục nhỏ như `VIDEO_PLAYERS` hoặc `DATING` có CI rộng, nên nên đọc thận trọng.

#### **5. Kiểm tra độ bền (D6)**

Nhóm lặp lại phân tích khi đổi: ngưỡng "mới" (30, 90 hoặc 180 ngày), cách phân tầng (thêm `Size_group`), phạm vi mẫu (loại ứng dụng kích thước "Varies") và ngưỡng phổ biến (100.000, 1.000.000 hoặc 10.000.000).

In [ ]:
# D6: đổi ngưỡng "mới", cách phân tầng, phạm vi mẫu và ngưỡng phổ biến.
variants = [
    ("Chính: ≤90 ngày, Category×Type", dict(d=q2)),
    ("Ngưỡng mới ≤30 ngày", dict(d=q2, cutoff=30)),
    ("Ngưỡng mới ≤180 ngày", dict(d=q2, cutoff=180)),
    ("Phân tầng thêm Size_group", dict(d=q2, strata=("Category", "Type", "Size_group"))),
    ("Loại app có Size 'Varies'", dict(d=q2[q2["Size_group"].ne("Varies")])),
    ("Phổ biến: Installs ≥ 100.000", dict(d=q2, thr=100_000)),
    ("Phổ biến: Installs ≥ 10.000.000", dict(d=q2, thr=10_000_000)),
]
rows = []
for label, kw in variants:
    r = q2_estimate(**kw)
    rows.append({"Biến thể": label, "n": r["n"], "App dùng": r["App dùng"], "Chênh thô": r["Chênh thô"],
                 "Chênh sau phân tầng": r["Chênh sau phân tầng"],
                 "CI 95%": f"[{r['CI phân tầng'][0]:.3f}; {r['CI phân tầng'][1]:.3f}]"})
q2_robust = pd.DataFrame(rows)
with pd.option_context("display.width", 200, "display.max_columns", None):
    display(q2_robust.round(3))

- Chiều của kết quả không đổi ở cả 7 biến thể: chênh lệch sau phân tầng luôn dương và CI luôn nằm hẳn trên 0, từ **+0,164** đến **+0,292**.
- Độ lớn thay đổi theo lựa chọn. Thêm `Size_group` vào tầng làm chênh lệch giảm từ 0,281 xuống **0,208**, nghĩa là một phần mối liên hệ ban đầu đi cùng kích thước ứng dụng. Ngưỡng phổ biến cao hơn (≥10 triệu) cho chênh lệch nhỏ hơn (**+0,164**), vì ngưỡng khó đạt hơn nên tỷ lệ chung thấp hơn.
- Kết luận định tính (ứng dụng cập nhật gần đây phổ biến hơn) bền; con số cụ thể nhạy với ngưỡng và biến kiểm soát.

#### **6. Trả lời câu hỏi và giới hạn**

**Trả lời:** Có. Ứng dụng cập nhật trong 90 ngày gần nhất có tỷ lệ đạt từ 1 triệu lượt cài cao hơn **0,281** (CI 95% [0,262; 0,297]) so với ứng dụng cũ hơn, khi so sánh trong cùng danh mục và loại Free hoặc Paid (tỷ lệ thô 49,9% so với 20,8%). Mối liên hệ dương ở cả 20 danh mục lớn và ở cả 7 biến thể kiểm tra, dao động từ 0,164 đến 0,292; khi kiểm soát thêm kích thước, ước lượng còn 0,208. Spearman giữa số ngày từ lần cập nhật cuối và `Installs` là −0,332. Với nhà phát triển, duy trì cập nhật đi cùng khả năng đạt quy mô lớn, nhưng chưa chứng minh được cập nhật gây ra điều đó.

**Giới hạn:**
- Đây là mối liên hệ quan sát, và có thể đảo chiều: ứng dụng đông người dùng có nhiều nguồn lực để cập nhật thường xuyên hơn.
- `Last Updated` chỉ là ngày cập nhật cuối, không cho biết tần suất cập nhật. Ngày tham chiếu 08/08/2018 là xấp xỉ ngày thu thập, nên "độ mới" phụ thuộc thời điểm cào dữ liệu.
- `Installs` là cận dưới theo ngưỡng, và dữ liệu không có ứng dụng đã bị gỡ khỏi cửa hàng (thiên lệch sống sót).
- Một số danh mục chỉ có 150–400 ứng dụng, nên CI theo danh mục khá rộng; ngưỡng "mới ≤90 ngày" là lựa chọn của nhóm.

# **III. Summary and reflection**

Ba mục đầu tổng hợp kết quả của cả dự án, từ khám phá và tiền xử lý đến phân tích và học máy. Mục phản tư cá nhân ghi lại phần việc của người viết.

### **1. Key findings**

1. **Tình trạng thiếu điểm tập trung ở các mốc cài thấp.** Trong 10.841 bản ghi gốc, **1.474 bản ghi (13,60%)** thiếu `Rating`. Tỷ lệ thiếu cao hơn ở các bậc lượt cài thấp, nên các phân tích cần điểm đánh giá chỉ phản ánh một phần của mẫu. Bảng ứng dụng còn **10.357 bản ghi** sau khi loại dòng lệch cột và bản sao hoàn toàn.
2. **Kết quả bất ngờ nhất — Nhóm giá trên 2 USD đạt mốc lượt cài thường xuyên hơn.** Trong 536 cặp ứng dụng–danh mục đủ điều kiện ở câu 2.1, tỷ lệ đạt cận dưới 10.000 lượt cài là **47,27%** ở nhóm trên 2 USD và **25,33%** ở nhóm đến 2 USD. Thứ tự này xuất hiện trong cả sáu danh mục.
3. **Lượt cài và điểm đánh giá có liên hệ dương yếu.** Ở câu 2.2, trên **5.934 ứng dụng miễn phí** có ít nhất 100 lượt đánh giá, Spearman là **0,1626**, với khoảng tin cậy bootstrap 95% **[0,1365; 0,1876]**. Khi đổi ngưỡng tối thiểu sang 50 và 200, hệ số lần lượt là **0,1606** và **0,1576**.

### **2. Limitations**

- **Dữ liệu và phép đo.** Bộ dữ liệu phản ánh một phần Google Play vào năm 2018. `Installs` ghi cận dưới của khoảng cài đặt, `Price` ghi giá niêm yết khi thu thập và `Last Updated` chỉ ghi lần cập nhật cuối. Một tên ứng dụng có thể xuất hiện ở nhiều bản ghi, nên cần thận trọng khi dùng tên để chọn bản ghi đại diện hoặc chia nhóm.
- **Phạm vi mẫu.** Câu 2.1 dựa trên sáu danh mục có đủ quan sát ở cả hai mức giá. Câu 2.2 giới hạn ở ứng dụng miễn phí có điểm sao và đủ lượt đánh giá. Các mẫu theo ngưỡng lượt đánh giá có chung nhiều ứng dụng, nên phép kiểm tra độ bền phản ánh độ nhạy với điều kiện lọc trong cùng nguồn dữ liệu.
- **Diễn giải phân tích quan sát.** Giá, điểm sao, ngày cập nhật và lượng cài đều được ghi nhận trong cùng đợt thu thập. Thời gian có mặt trên cửa hàng, quảng bá và mức độ nhận diện thương hiệu có thể ảnh hưởng đến những mối liên hệ quan sát được. Phân tầng giúp so sánh các nhóm tương đồng theo những thuộc tính đã ghi nhận.
- **Phạm vi đánh giá mô hình.** Việc chia theo nhóm tên ứng dụng giữ các dòng cùng tên trong một tập duy nhất. Tập huấn luyện và tập kiểm tra vẫn thuộc cùng đợt thu thập. Tên, dung lượng và yêu cầu Android có thể đã thay đổi sau khi phát hành, nên hiệu năng dự báo trước phát hành hoặc trên một giai đoạn mới vẫn cần được kiểm chứng.

### **3. Future work**

- **Xây dựng dữ liệu theo dõi ứng dụng qua nhiều thời điểm.** Thu thập mã định danh, ngày phát hành, lịch sử giá, khuyến mãi, các lần cập nhật, lượt cài và điểm sao. Mã định danh giúp liên kết đúng ứng dụng khi tên hoặc danh mục thay đổi.
- **Đo thay đổi sau khi điều chỉnh giá hoặc phát hành phiên bản.** So sánh biến động lượt cài và điểm sao trước và sau từng sự kiện với các ứng dụng tương đồng trong cùng thời gian. Theo dõi thêm thời gian từ lúc phát hành để tách giai đoạn mới ra mắt khỏi giai đoạn đã có người dùng ổn định.
- **Kiểm tra độ phủ của điểm đánh giá.** Trên mẫu mới, thống kê tỷ lệ có điểm theo mốc lượt cài và danh mục, rồi lặp lại câu 2.2 dưới các ngưỡng lượt đánh giá. Việc này giúp đánh giá khả năng áp dụng kết quả cho nhóm ứng dụng còn ít phản hồi.
- **Kiểm định mô hình trên các đợt phát hành mới hơn.** Dùng thuộc tính được lưu tại thời điểm phát hành và chọn một khoảng theo dõi cố định để xác định ứng dụng đạt một triệu lượt cài. Huấn luyện trên các đợt trước, kiểm tra ở các đợt sau và báo cáo hiệu năng cùng tỷ lệ lỗi theo nhóm.

### **4. Individual reflection**

#### **Phạm Hoàng Tuấn Anh**

- **Khó khăn.** Ở câu 2.1, cùng một tên ứng dụng có thể xuất hiện nhiều lần hoặc thuộc nhiều danh mục, nên em cần chọn đơn vị quan sát trước khi tính tỷ lệ. Ở câu 2.2, em cần phân biệt số lượt đánh giá tối thiểu của mỗi ứng dụng với số ứng dụng thực sự còn lại trong mẫu.
- **Cách xử lý.** Em dùng một bản ghi cho mỗi cặp ứng dụng–danh mục ở câu 2.1 và một bản ghi cho mỗi tên ứng dụng ở câu 2.2. Em đọc kết quả cùng bảng đếm sau từng bước lọc, rồi so sánh hệ số tương quan và khoảng tin cậy qua ba ngưỡng lượt đánh giá.
- **Điều học được.** Em hiểu vì sao phải đọc tỷ lệ gộp cùng tỷ lệ theo từng danh mục và cỡ nhóm. Qua câu 2.2, em học cách phân biệt độ mạnh của tương quan với độ bất định của ước lượng. Khoảng tin cậy nằm phía dương vẫn có thể đi cùng một hệ số tương quan nhỏ.
- **Sử dụng LLM.** Em sử dụng GPT như một người hướng dẫn trong quá trình làm bài. Công cụ giúp em đọc yêu cầu, cân nhắc các cách đặt câu hỏi, hiểu tiêu chí D3–D6 và phân biệt ngưỡng lượt đánh giá với cỡ mẫu. GPT cũng hỗ trợ đối chiếu số liệu, rà soát nhận xét, chỉnh cách trình bày mã và tổng hợp kết quả đã có.